# Controlled MovieLens LightGCN Run

This notebook inspects the first run on real MovieLens edges. It is an integration check, not a final experiment: pair users are retained, context users are sampled, and validation/test labels are not used.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
output_dir = project_root / "outputs/lightgcn_subset"
if not (output_dir / "summary.json").exists():
    raise FileNotFoundError(
        "Run scripts/train_lightgcn_subset.py before this notebook."
    )

## 1. Confirm the scope

The focus users come from selected dissimilar pairs. Other users provide collaborative graph context. Movies enter the graph only when at least one retained user has a positive training edge to them.

In [ ]:
summary = json.loads((output_dir / "summary.json").read_text(encoding="utf-8"))
summary

## 2. Inspect learning rather than recommendation quality

The fixed diagnostic batch contains training triples. A lower loss confirms optimization, but it is not NDCG, Recall, or evidence of generalization.

In [ ]:
history = pd.read_csv(output_dir / "training_history.csv.gz")
history.iloc[::max(1, len(history) // 5)]

## 3. Load final user and movie embeddings

The NPZ artifact stores final propagated embeddings together with original MovieLens IDs. This avoids confusing embedding row indices with source IDs.

In [ ]:
artifact = np.load(output_dir / "final_embeddings.npz")
user_ids = artifact["user_ids"]
user_embeddings = artifact["user_embeddings"]
movie_ids = artifact["movie_ids"]
movie_embeddings = artifact["movie_embeddings"]

print("Users:", user_embeddings.shape)
print("Movies:", movie_embeddings.shape)

## 4. Produce individual scores for one pair

These are raw full-catalog scores within the subset. Do not rank them yet: movies observed by either member must first be removed.

In [ ]:
pairs = pd.read_csv(output_dir / "focus_pairs.csv.gz")
pair = pairs.iloc[0]
user_a_index = int(np.searchsorted(user_ids, pair.userA))
user_b_index = int(np.searchsorted(user_ids, pair.userB))
score_a = movie_embeddings @ user_embeddings[user_a_index]
score_b = movie_embeddings @ user_embeddings[user_b_index]

pair_scores = pd.DataFrame(
    {"movieId": movie_ids, "scoreA": score_a, "scoreB": score_b}
)
pair_scores.sort_values("scoreA", ascending=False).head()

## Next step

Build validation candidate sets from warm movies unseen in training by both users, restrict held-out positives to the subset catalogue, and compare average with conflict-aware ranking using NDCG@10, Recall@10, satisfaction gap, and catalogue coverage.